# SmallWorld-MJ World Model Homework\n\nThis notebook runs the reward-free SmallWorld-MJ world-model homework: MuJoCo dataset generation, one-step Pendulum baseline, 10-task solution/student smoke training, official 10/90 evaluation, and rollout visualization.

In [ ]:
from pathlib import Path\nimport os, subprocess, sys\n\nCOURSE_REPO_URL = 'https://github.com/WeijieLai1024/EEC289A_WorldModel-Homework.git'\nCOURSE_REPO_BRANCH = 'main'\nCOURSE_REPO_DIR = Path('/content/world_model_course_repo') if Path('/content').exists() else Path.home() / 'world_model_course_repo'\nPYTHON = sys.executable\n\ndef run(cmd, cwd=None):\n    print('+', ' '.join(map(str, cmd)))\n    return subprocess.run(list(map(str, cmd)), cwd=cwd, check=True)\n\ncwd = Path.cwd()\nif (cwd / 'configs' / 'smallworld_mj.yaml').exists():\n    COURSE_REPO_DIR = cwd\nelif not COURSE_REPO_DIR.exists():\n    run(['git', 'clone', '--branch', COURSE_REPO_BRANCH, COURSE_REPO_URL, COURSE_REPO_DIR])\nos.chdir(COURSE_REPO_DIR)\nprint('repo:', COURSE_REPO_DIR)\n

In [ ]:
!{PYTHON} -m pip install -q -r configs/colab_requirements.txt\nimport mujoco, torch\nprint('mujoco:', mujoco.__version__)\nprint('torch:', torch.__version__)\nprint('cuda available:', torch.cuda.is_available())\n

## Inspect Frozen MuJoCo Tasks

In [ ]:
!{PYTHON} inspect_mj_tasks.py --config configs/smallworld_mj.yaml

## Generate Smoke Dataset

In [ ]:
!{PYTHON} generate_mj_dataset.py --config configs/smallworld_mj.yaml --taskpack smallworld_all --profile smoke --output-dir artifacts/smoke/mj_data

## Run One-Step Pendulum Baseline

In [ ]:
!{PYTHON} run_experiment.py --config configs/pendulum_baseline.yaml --task simple_pendulum --model baseline_mlp --dataset-dir artifacts/smoke/mj_data --local-smoke --output-dir artifacts/smoke/pendulum_baseline

## Train Staff Solution Smoke

In [ ]:
!{PYTHON} run_experiment.py --config configs/solution.yaml --taskpack smallworld_all --model solution --dataset-dir artifacts/smoke/mj_data --local-smoke --output-dir artifacts/smoke/solution

## Evaluate Test/OOD and Visualize

In [ ]:
!{PYTHON} eval.py --checkpoint-dir artifacts/smoke/solution/best_checkpoint --dataset-dir artifacts/smoke/mj_data --split test --warmup 10 --horizon 90 --output-dir artifacts/smoke/eval_test\n!{PYTHON} eval.py --checkpoint-dir artifacts/smoke/solution/best_checkpoint --dataset-dir artifacts/smoke/mj_data --split ood --warmup 10 --horizon 90 --output-dir artifacts/smoke/eval_ood\n!{PYTHON} visualize_rollout.py --checkpoint-dir artifacts/smoke/solution/best_checkpoint --dataset-dir artifacts/smoke/mj_data --task bouncing_ball --split test --output-dir artifacts/smoke/viz

The video `artifacts/smoke/viz/smallworld_mj_rollout.mp4` compares the MuJoCo target trajectory with the model's predicted ghost trajectory. It is not a policy video; no actor or reward is used.